In [1]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler

# --- Load & Preprocess Expression ---
def load_expression(file, n_genes):
    df = pd.read_csv(file, sep='\t', index_col=0).T
    df = df.apply(pd.to_numeric, errors='coerce')
    non_null_cols = df.columns[df.isnull().sum() == 0]  # keep only fully non-null columns
    df = df[non_null_cols[:n_genes]]  # select top N non-null genes
    df = pd.DataFrame(StandardScaler().fit_transform(df), index=df.index, columns=df.columns)
    return df.astype(np.float32)

# --- Load & Preprocess Methylation ---
def load_methylation(file, n_cpgs):
    df = pd.read_csv(file, sep='\t', index_col=0)
    df = df.apply(pd.to_numeric, errors='coerce')
    non_null_rows = df.index[df.isnull().sum(axis=1) == 0]  # keep only fully non-null rows
    df = df.loc[non_null_rows[:n_cpgs]]  # select top N non-null CpGs
    df = df.T  # Transpose so samples are rows
    df = pd.DataFrame(StandardScaler().fit_transform(df), index=df.index, columns=df.columns)
    return df.astype(np.float32)

# --- File paths ---
meth_file = "HumanMethylation450.tsv"
expr_file = "HiSeqV2_PANCAN.tsv"

# --- Load Data ---
expr_df = load_expression(expr_file, n_genes=500)
meth_df = load_methylation(meth_file, n_cpgs=500)

# --- Done ---
print(f"Loaded expression data: {expr_df.shape}")
print(f"Loaded methylation data: {meth_df.shape}")


✅ Loaded expression data: (1129, 500)
✅ Loaded methylation data: (907, 500)


In [2]:
# --- Align methylation and expression by sample ID ---
common_samples = expr_df.index.intersection(meth_df.index)
print(f"Common sample IDs found: {len(common_samples)}")

# Subset both datasets to common samples only
expr_df_aligned = expr_df.loc[common_samples].sort_index()
meth_df_aligned = meth_df.loc[common_samples].sort_index()

Common sample IDs found: 856


In [3]:
# Align and clean expression and methylation data

# Step 1: Align on common sample IDs
common_samples = expr_df.index.intersection(meth_df.index)
expr_df_aligned = expr_df.loc[common_samples].sort_index()
meth_df_aligned = meth_df.loc[common_samples].sort_index()

# Step 2: Drop columns with any NaNs from both
cols_with_na = meth_df_aligned.columns[meth_df_aligned.isna().any()].tolist()
print("Dropping {} columns with NaNs in methylation".format(len(cols_with_na)))
meth_df_aligned = meth_df_aligned.drop(columns=cols_with_na)
expr_df_aligned = expr_df_aligned.drop(columns=cols_with_na, errors='ignore')  # Optional

# Step 3: Save
expr_df_aligned.to_csv("expression_input.csv")
meth_df_aligned.to_csv("methylation_input.csv")


Dropping 0 columns with NaNs in methylation
